# IRIS Step 5b: engine-managed CUDA graphs

Select a GPU runtime. This experimental capture path is not yet GPU-qualified. Run the setup and direct-engine check first. Only proceed to the server check after the direct check passes. Download the evidence even on failure. See [CAPTURE.md](https://github.com/Artifex2002/mstar/blob/codex/iris/scripts/iris/CAPTURE.md).


In [ ]:
%pip install -q uv


In [ ]:
%%bash
set -euo pipefail
if [ ! -e /content/mstar-iris ]; then
  git clone --branch codex/iris --single-branch https://github.com/Artifex2002/mstar.git /content/mstar-iris
fi
cd /content/mstar-iris
git switch codex/iris
git pull --ff-only origin codex/iris
if [ ! -e .venv ]; then
  uv venv --python 3.12 .venv
fi
export PATH="$HOME/.cargo/bin:$PWD/.venv/bin:$PATH"
if ! command -v rustup >/dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
uv pip install --python .venv/bin/python --torch-backend=auto \
  -r scripts/iris/requirements-cuda-tests.txt -e '.[iris,metrics]' maturin
env -u CONDA_PREFIX VIRTUAL_ENV="$PWD/.venv" \
  .venv/bin/maturin develop --release --locked -m rust/Cargo.toml
.venv/bin/python -c 'import mstar_rust; print(mstar_rust.__version__)'

mkdir -p .ci-results/iris
if [ ! -e .ci-results/iris/reference ]; then
  git clone https://github.com/eloialonso/iris.git .ci-results/iris/reference
fi
git -C .ci-results/iris/reference checkout --detach 24326aaaa283c527f42b89b44cfdecf2665a7a16


## Direct engine qualification
This regenerates official CUDA traces, captures every declared engine bucket, and audits actual replay calls. Capture sizing and first kernel builds can take several minutes.


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
mkdir -p .ci-results/iris
OMP_NUM_THREADS=2 .venv/bin/python -m scripts.iris.qualify_capture \
  --download-checkpoint --output .ci-results/iris/capture-check-01 \
  2>&1 | tee .ci-results/iris/capture-check-01.log


## Captured full-server qualification
Run only after the previous cell reports PASSED. Use new directory names on reruns and update the download cell too.


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
mkdir -p .ci-results/iris
OMP_NUM_THREADS=2 .venv/bin/python -m scripts.iris.qualify_server \
  --config configs/iris_capture.yaml --download-checkpoint \
  --startup-timeout 1800 --request-timeout 600 \
  --output .ci-results/iris/capture-server-check-01 \
  2>&1 | tee .ci-results/iris/capture-server-check-01.log


## Download reports and logs
This collects whichever reports/logs exist, including after a failure.


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

root = Path("/content/mstar-iris/.ci-results/iris")
archive = Path("/content/iris-capture-check-01.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output:
    for directory in ("capture-check-01", "capture-server-check-01"):
        for name in ("report.json", "server.log"):
            path = root / directory / name
            if path.is_file():
                output.write(path, f"{directory}/{name}")
    for name in ("capture-check-01.log", "capture-server-check-01.log"):
        path = root / name
        if path.is_file():
            output.write(path, name)
files.download(str(archive))
